# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [1]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import generate_word_wfst, parse_lexicon, generate_symbol_tables, generate_phone_wfst
import math
import time

def create_wfst(lexicon, word_table, phone_table, state_table, n_states=3):
    f = fst.Fst('log')
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    loop_state = f.add_state()
    f.set_start(loop_state)
    f.set_final(loop_state)

    for word, phones in lexicon.items():
        word_id = word_table.find(word)
        current_state = loop_state

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                state_sym = f"{phone}_{i}"
                in_label = state_table.find(state_sym)

                sl_weight = fst.Weight('log', -math.log(0.5))
                f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))

                next_state = f.add_state()
                fw_weight = fst.Weight('log', -math.log(0.5))

                if phone_idx == len(phones) - 1 and i == n_states:
                    out_label = word_id
                else:
                    out_label = 0

                f.add_arc(current_state, fst.Arc(in_label, out_label, fw_weight, next_state))
                current_state = next_state

        f.add_arc(current_state, fst.Arc(0, 0, fst.Weight.One(f.weight_type()), loop_state))

    return f


def read_transcription(wav_file):
    transcription_file = os.path.splitext(wav_file)[0] + '.txt'
    with open(transcription_file, 'r') as f:
        transcription = f.readline().strip()
    return transcription


lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)
f = create_wfst(lex, word_table, phone_table, state_table)

# --- (B) WFST memory analysis ---
num_states = f.num_states()
num_arcs = sum(1 for s in f.states() for _ in f.arcs(s))
print(f"=== WFST Size ===")
print(f"  States : {num_states}")
print(f"  Arcs   : {num_arcs}\n")

total_substitutions = 0
total_deletions = 0
total_insertions = 0
total_words = 0
total_forward_computations = 0
total_decode_time = 0.0
total_backtrace_time = 0.0

for wav_file in glob.glob('/group/teaching/asr/labs/recordings/*.wav'):

    decoder = MyViterbiDecoder(f, wav_file)

    # --- (A) Timing ---
    t0 = time.perf_counter()
    decoder.decode()
    decode_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    (state_path, words) = decoder.backtrace()
    backtrace_time = time.perf_counter() - t0

    total_decode_time += decode_time
    total_backtrace_time += backtrace_time
    total_forward_computations += decoder.forward_computation_count

    words_str = ' '.join(words)
    transcription = read_transcription(wav_file)
    error_counts = wer.compute_alignment_errors(transcription, words_str)
    word_count = len(transcription.split())

    total_substitutions += error_counts[0]
    total_deletions += error_counts[1]
    total_insertions += error_counts[2]
    total_words += word_count

    print(f"File: {os.path.basename(wav_file)}")
    print(f"  Hypothesis           : {words_str}")
    print(f"  Reference            : {transcription}")
    print(f"  Errors (S, D, I)     : {error_counts}")
    print(f"  Decode time          : {decode_time:.4f}s")
    print(f"  Backtrace time       : {backtrace_time:.4f}s")
    print(f"  Forward computations : {decoder.forward_computation_count}\n")

# --- Final summary ---
overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words

print(f"=== Overall Results ===")
print(f"  WER                        : {overall_wer:.2%}")
print(f"  Total substitutions        : {total_substitutions}")
print(f"  Total deletions            : {total_deletions}")
print(f"  Total insertions           : {total_insertions}")
print(f"  Total decode time          : {total_decode_time:.4f}s")
print(f"  Total backtrace time       : {total_backtrace_time:.4f}s")
print(f"  Total time                 : {total_decode_time + total_backtrace_time:.4f}s")
print(f"  Total forward computations : {total_forward_computations}")
print(f"  WFST states                : {num_states}")
print(f"  WFST arcs                  : {num_arcs}")

ModuleNotFoundError: No module named 'openfst_python'